# 01 · Data audit: sources, coverage and cleaning

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

from boxbox.config import load_config
from boxbox.paths import artifact_paths, dataset_paths
from boxbox.preprocessing.build import read_lap_dataset

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4)
COLOURS = {"SOFT": "#c93a55", "MEDIUM": "#b07c00", "HARD": "#3b6fb0", "UNKNOWN": "#77766f"}
subset, full = load_config("subset"), load_config("full")

Which source actually supplied each season (recorded at download time). The F1 live-timing archive returned HTTP 401 for most seasons from this network, so FastF1 is used where it works (2021-22) and the pipeline falls back to OpenF1 (2023+) and a Kaggle FastF1 export (2018-20). Lap times, positions and pit stops always come from the Ergast-format Kaggle dump.

In [ ]:
sources = json.loads(dataset_paths().sources_record.read_text())
pd.DataFrame(sources).T[['round_number', 'tables_source', 'tyre_source']]

In [ ]:
from boxbox.reporting.eda import race_overview, exclusion_table
full_laps = read_lap_dataset(full).laps
race_overview(full_laps)

Every row is kept; `exclusion_reason` records the first rule that removed a lap from modelling.

In [ ]:
exclusion_table(full_laps)

Cross-check: tyre-layer lap times against the Ergast backbone (should agree to the millisecond for authentic data).

In [ ]:
rows = []
for season in subset.seasons:
    tyre = pd.read_parquet(dataset_paths().interim_table(season, 'tyre_layer'))
    ergast = pd.read_parquet(dataset_paths().interim_table(season, 'laps'))
    m = tyre.merge(ergast, on=['driver_id', 'lap'], suffixes=('_tyre', '_ergast'))
    rows.append({'season': season, 'tyre_source': sources[str(season)]['tyre_source'], 'laps': len(m),
                 'match_within_10ms': ((m.lap_time_s_tyre - m.lap_time_s_ergast).abs() < 0.01).mean()})
pd.DataFrame(rows)

In [ ]:
laps = read_lap_dataset(subset).laps
fig, ax = plt.subplots()
laps.groupby('race_id')['is_clean'].mean().plot.bar(ax=ax, color='#2a78d6')
ax.set(ylabel='share of clean laps', title='Clean-lap share per race (subset)');